## 🌳 Random Forest – Brief Overview

**Random Forest** is an **ensemble machine learning algorithm** that builds multiple **decision trees** and combines their predictions to improve accuracy and reduce overfitting.

### 🔹 How it works
- Creates many decision trees using **bootstrap sampling** (random rows with replacement)
- At each split, a tree considers only a **random subset of features**
- Final prediction is:
  - **Classification** → majority vote
  - **Regression** → average of predictions

### 🔹 Why it works well
- Reduces overfitting compared to a single decision tree
- Handles **non-linear relationships** effectively
- Works well with minimal feature engineering
- Provides **feature importance**

### 🔹 Key hyperparameters
- `n_estimators` – number of trees
- `max_depth` – depth of each tree
- `max_features` – features considered per split
- `min_samples_split`, `min_samples_leaf`

### 🔹 Pros & Cons
**Pros**
- High accuracy
- Robust to noise and outliers
- Handles large datasets well

**Cons**
- Slower and memory-intensive
- Less interpretable than single trees

### 🔹 Use cases
- Classification and regression problems
- When data has complex patterns and interactions


In [0]:
import numpy as np
import pandas as pd
from collections import Counter


In [0]:
%run "./Decision Trees"


In [0]:
class RandomForestRegressor:

    def __init__(
        self,
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        random_state=None
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

        if random_state is not None:
            np.random.seed(random_state)

    def _bootstrap_sample(self, X, y):
        n_samples = len(X)
        indices = np.random.choice(n_samples, size=n_samples, replace=True)

        X_sample = X.iloc[indices] if hasattr(X, "iloc") else X[indices]
        y_sample = y.iloc[indices] if hasattr(y, "iloc") else y[indices]

        return X_sample, y_sample

    def _train_single_tree(self, X, y):
        X_sample, y_sample = self._bootstrap_sample(X, y)

        tree = DecisionTreeRegressor(
            max_depth=self.max_depth,
            min_samples_split=self.min_samples_split,
            min_samples_leaf=self.min_samples_leaf,
            max_features=self.max_features
        )

        tree.fit(X_sample, y_sample)
        return tree

    def fit(self, X, y):
        self.trees = []

        for _ in range(self.n_estimators):
            tree = self._train_single_tree(X, y)
            self.trees.append(tree)

    def predict(self, X):
        # Collect predictions from all trees
        all_predictions = []

        for tree in self.trees:
            preds = tree.predict(X)
            all_predictions.append(preds)

        # Shape → (n_trees, n_samples)
        all_predictions = np.array(all_predictions)

        # Majority vote per sample
        final_predictions = []
        for i in range(all_predictions.shape[1]):
            vote = Counter(all_predictions[:, i]).most_common(1)[0][0]
            final_predictions.append(vote)

        return np.array(final_predictions)


In [0]:
# Feature matrix
X = pd.DataFrame({
    "area_sqft": [600, 750, 800, 900, 1100, 1200, 1400, 1500, 1600, 1800,
                  650, 700, 850, 1000, 1300, 1700, 2000, 2200],
    "bedrooms": [1, 2, 2, 2, 3, 3, 3, 3, 4, 4,
                 1, 1, 2, 2, 3, 4, 4, 5],
    "age_of_house": [20, 15, 10, 12, 8, 6, 5, 7, 4, 3,
                     25, 22, 18, 14, 9, 6, 2, 1],
    "distance_to_city_km": [15, 12, 10, 9, 8, 7, 6, 6, 5, 4,
                            18, 16, 11, 9, 7, 5, 3, 2]
})

# Continuous target
y = pd.Series(
    [30, 38, 42, 45, 55, 60, 70, 75, 82, 90,
     28, 32, 40, 48, 62, 85, 105, 120],
    name="price_lakhs"
)


In [0]:
rf = RandomForestRegressor(
    n_estimators=50,
    max_depth=6,
    max_features="sqrt",
    random_state=42
)

rf.fit(X, y)

preds = rf.predict(X)

print("Predicted prices:", preds)
print("Actual prices:", y.values)


In [0]:
# New unseen data point
new_house = pd.DataFrame({
    "area_sqft": [1450],
    "bedrooms": [3],
    "age_of_house": [6],
    "distance_to_city_km": [6]
})

# Predict price
predicted_price = rf.predict(new_house)

print("Predicted house price (in lakhs):", predicted_price[0])
